# RAG_test

In [ ]:
import io
import warnings
import fitz
from langchain.chains import RetrievalQA, StuffDocumentsChain, LLMChain
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ml_server_embedding import get_embeddings
from llm_config import get_llm

warnings.filterwarnings("ignore", category=DeprecationWarning)


class PDFVectorStore:

    def __init__(self, pdf_bytes: bytes, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.embeddings = get_embeddings()
        self.index: FAISS = self.create_index(self.read_chunks(pdf_bytes))

    # ── Step 1: Read & Chunk ──────────────────────────────────────────────────

    def read_chunks(self, pdf_bytes: bytes) -> list[Document]:
        with fitz.open(stream=io.BytesIO(pdf_bytes), filetype="pdf") as doc:
            full_text = "".join(page.get_text() for page in doc)

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ".", " ", ""]
        )

        return [
            Document(page_content=chunk, metadata={"chunk_index": i})
            for i, chunk in enumerate(splitter.split_text(full_text))
        ]

    # ── Step 2: Build FAISS Index ─────────────────────────────────────────────

    def create_index(self, documents: list[Document]) -> FAISS:
        return FAISS.from_documents(documents, self.embeddings)

    # ── Step 3: Build RetrievalQA Chain ───────────────────────────────────────

    def build_rag_chain(self) -> RetrievalQA:
        prompt = PromptTemplate(
            input_variables=["context", "question"],
            template="""
            Use the following context to answer the question.
            If you don't know the answer, say "I don't know" — do not make up an answer.

            Context: {context}

            Question: {question}

            Answer:
            """
        )

        llm = get_llm()

        stuff_chain = StuffDocumentsChain(
            llm_chain=LLMChain(llm=llm, prompt=prompt),
            document_variable_name="context"
        )

        return RetrievalQA(
            combine_documents_chain=stuff_chain,
            retriever=self.index.as_retriever(search_kwargs={"k": 3}),
            return_source_documents=True
        )

    # ── Step 4: Ask ───────────────────────────────────────────────────────────

    def ask(self, question: str) -> dict:
        try:
            result = self.build_rag_chain().invoke({"query": question})
            return {
                "answer": result["result"],
                "sources": [doc.metadata for doc in result["source_documents"]]
            }
        except Exception as e:
            print(f"Error: {type(e).__name__}: {e}")
            return {"answer": None, "sources": []}


# ── Usage ─────────────────────────────────────────────────────────────────────

with open("sample_rag_test.pdf", "rb") as f:
    pdf_bytes = f.read()

store = PDFVectorStore(pdf_bytes)

response = store.ask("Explain about AI goverance process?")
print("Answer:", response["answer"])
print("Sources:", response["sources"])